# ShopKart cleaning pipeline: raw to shopkart_clean.csv

**Video 6.3 to 6.6 (end-to-end)** | ShopKart project

**Goal:** One notebook that runs every cleaning and merge step from Module 6 in order and writes shopkart_clean.csv - the file Modules 8 and 9 use (the Module 10 capstone pulls the same cleaned data straight from shopkart.db with SQL).

### How to open in Colab
1. Go to colab.research.google.com → **File → Upload notebook** → choose this `.ipynb` file.
2. Click the **folder icon** on the left sidebar (Files panel) and drag in the data files: `orders_raw.csv`, `products.csv`, `customers.csv`.
3. Run cells one by one with **Shift + Enter**.

> Files you upload to the Files panel disappear when the runtime resets. If you see `FileNotFoundError`, upload them again.

Run with **Runtime → Run all**. Every step is explained in videos 6.3 (cleaning), 6.4 (new columns) and 6.6 (merge).

In [1]:
import pandas as pd

df = pd.read_csv("orders_raw.csv")
products = pd.read_csv("products.csv")
customers = pd.read_csv("customers.csv")
print("Raw orders:", df.shape)

Raw orders: (5055, 12)


## 1. Clean the orders

In [2]:
# 1. Remove 55 exact duplicate rows
df = df.drop_duplicates()

# 2. Standardise city names (27 spellings -> 10 cities)
df["city"] = df["city"].astype(str).str.strip().str.title()
df["city"] = df["city"].replace({"New Delhi": "Delhi", "Bangalore": "Bengaluru", "Bombay": "Mumbai",
                                 "Madras": "Chennai", "Calcutta": "Kolkata"})

# 3. Prices like "₹1,299" -> 1299
df["unit_price"] = (df["unit_price"].astype(str)
                    .str.replace("₹", "", regex=False)
                    .str.replace(",", "", regex=False)
                    .astype(float).astype(int))

# 4. Two date formats -> real dates. Exact formats one at a time
#    (format="mixed", dayfirst=True would silently flip ISO dates like 2025-01-02 to 1 Feb)
s_ = df["order_date"].astype(str)
df["order_date"] = pd.to_datetime(s_, format="%Y-%m-%d", errors="coerce").fillna(
                   pd.to_datetime(s_, format="%d-%m-%Y", errors="coerce"))

# 5. Payment mode: one spelling for UPI, blanks -> "Unknown"
df["payment_mode"] = df["payment_mode"].replace({"upi": "UPI"}).fillna("Unknown")

# 6. Blank discount means no discount
df["discount"] = df["discount"].astype(float).fillna(0.0)

# 7. Returned: Yes/No -> Y/N
df["returned"] = df["returned"].replace({"Yes": "Y", "No": "N"})

# 8. Quantity 99 is a typo for 1 (the three 10-Smartphone bulk orders are real - keep them)
df.loc[df["quantity"] == 99, "quantity"] = 1

# 9. rating: blanks stay blank on purpose ("not rated" is real information)
print("Clean orders:", df.shape)
print(df.isnull().sum()[lambda s: s > 0])

Clean orders: (5000, 12)
rating    260
dtype: int64


## 2. Merge product and customer details

In [3]:
df = df.merge(products[["product_id", "product_name", "category", "cost_price"]], on="product_id", how="left")
df = df.merge(customers[["customer_id", "gender", "age", "membership"]], on="customer_id", how="left")
assert len(df) == 5000, "Row count changed during merge!"
print("After merges:", df.shape)

After merges: (5000, 18)


## 3. Calculated columns

In [4]:
df["revenue"] = (df["quantity"] * df["unit_price"] * (1 - df["discount"])).round(2)
df["profit"] = df["revenue"] - df["quantity"] * df["cost_price"]
df["order_month"] = df["order_date"].dt.month
print(list(df.columns))

['order_id', 'order_date', 'customer_id', 'product_id', 'quantity', 'unit_price', 'discount', 'city', 'payment_mode', 'delivery_days', 'rating', 'returned', 'product_name', 'category', 'cost_price', 'gender', 'age', 'membership', 'revenue', 'profit', 'order_month']


## 4. Final checks and save

In [5]:
checks = {
    "5,000 rows": len(df) == 5000,
    "21 columns": df.shape[1] == 21,
    "no duplicate order_id": df["order_id"].is_unique,
    "10 cities": df["city"].nunique() == 10,
    "no quantity 99": (df["quantity"] != 99).all(),
    "all dates parsed": df["order_date"].notnull().all(),
    "only Y/N in returned": set(df["returned"]) == {"Y", "N"},
    "only rating has blanks": df.drop(columns="rating").isnull().sum().sum() == 0,
}
for name, ok in checks.items():
    print("OK  " if ok else "FAIL", name)

df.to_csv("shopkart_clean.csv", index=False)
print("Saved shopkart_clean.csv", df.shape)
print("Total revenue: ₹", round(df["revenue"].sum(), 2), "| Total profit: ₹", round(df["profit"].sum(), 2))

OK   5,000 rows
OK   21 columns
OK   no duplicate order_id
OK   10 cities
OK   no quantity 99
OK   all dates parsed
OK   only Y/N in returned
OK   only rating has blanks
Saved shopkart_clean.csv (5000, 21)
Total revenue: ₹ 14704649.7 | Total profit: ₹ 3140749.7


Download `shopkart_clean.csv` from the Files panel (right-click → Download) and keep it: Modules 8 and 9 start from this file. The Module 10 capstone pulls the same cleaned data straight from `shopkart.db` with SQL.